# Experiment — Simple Metrics
AA-01-03-E07 · Data visualization · AI with review

## Introduction

Simple metrics are numerical observations such as response length and elapsed time. They help check a specific requirement or describe execution, but do not establish factual correctness. A response can be short and wrong. HELM, first released as a preprint in 2022, illustrates the broader principle of evaluating several dimensions, including efficiency, rather than relying on accuracy alone ([paper](https://arxiv.org/abs/2211.09110)).

In this experiment, you will ask a data analyst for a caption describing three visitor counts in at most 20 words. You will count the words with Python, measure elapsed time around `get_completion()` and check the caption against the chart values yourself. You will explain why offline lookup time is not API latency, then try a shorter word limit without changing the data.

## 1. Understand the task

Read the input values below before running the request. This notebook includes all required setup and can be run independently. Companion files `simple-metrics-data.json` and `simple-metrics-reference.json` contain the same scenario and expected results.


**Downloaded package:** keep one private `.env` at the package root beside `COMPANION_VERSION`. The helper finds it automatically. When using this notebook alone, keep `.env` beside the notebook. The `get_completion()` interface is unchanged.

## Start here

This notebook includes its own setup. You do not need to run another notebook first. Start at the top and run each code cell in order, including the setup cells. To run everything, choose **Run → Run All Cells**. After restarting the kernel, run the setup again before running the experiment.

The default mode uses supplied examples and needs no API key. For OpenAI requests, follow the configuration instructions in this notebook and use a standard Python kernel.


## Provided setup — run once

Reuse `get_completion(system_prompt, user_prompt)` from Section 2.3. The shared notebook setup introduced in 3.2 adds supplied/live mode and records requests; its calling interface is unchanged here. Run the supplied helper cell, then the mode cell. You do not need to edit or retype the helper for live coding.

Default `"offline"` mode uses labelled supplied responses. For OpenAI, use a standard Python kernel, install `openai` and `python-dotenv`, and create `.env` beside the notebook containing `OPENAI_API_KEY=your_key` and `OPENAI_MODEL=your_model`. Set MODE to `"live"`. Browser Python supports the supplied examples. Never share a filled `.env` or print the key.

An edited request has no saved offline answer: the helper reports that no model was called. Do not evaluate that notice as a model response. Use live mode to test your changes.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
# Choose supplied examples or your own OpenAI account.
MODE = "offline"
if MODE == "live":
    configure_live()


In [ ]:
# A tiny zero-baseline text chart needs no plotting package.
visitors = {"Monday": 20, "Tuesday": 30, "Wednesday": 25}
for day, count in visitors.items():
    print(f"{day:9} {'█' * count} {count}")


## 2. Predict

What must the caption say accurately? Predict whether a useful caption can fit within 20 words. The supplied values support comparisons, not an explanation of why visits changed.

## 3. Run

Use the same request function introduced earlier. Only the persona, data and the few method-specific lines below are new. The original supplied responses are teaching examples, not recorded live model results.


In [ ]:
system_prompt = "You are a data analyst writing a chart caption. Use only supplied values. Write at most 20 words. Do not invent a cause."
user_prompt = f"Caption these visitor counts: {visitors}"


In [ ]:
# Supplied examples for these exact original requests; this does not call a model.
register_sample('You are a data analyst writing a chart caption. Use only supplied values. Write at most 20 words. Do not invent a cause.', "Caption these visitor counts: {'Monday': 20, 'Tuesday': 30, 'Wednesday': 25}", 'Tuesday had the most visitors (30), followed by Wednesday (25) and Monday (20).')


In [ ]:
from time import perf_counter

started = perf_counter()
response = get_completion(system_prompt, user_prompt)
elapsed = perf_counter() - started
word_count = len(response.split())
print(response)
print("Words:", word_count, "| Within 20-word limit:", word_count <= 20)
print("Elapsed seconds:", round(elapsed, 4), "| Mode:", MODE)


## 4. Check and explain

Check the numbers and highest day yourself. A caption can satisfy the word limit and still be wrong. Offline time is a local lookup; live duration includes network and local processing. One measurement is not typical model speed.

**Expected result:**

```json
{
  "word_limit": 20,
  "highest_day": "Tuesday",
  "highest_visitors": 30,
  "cause": "Not established by these values."
}
```


## 5. Try one change

Change the limit to 12 words and predict what can be removed without losing the key observation. Test the revised instruction live; do not score an offline “no saved response” notice as a caption.

Submit your prediction, the observed result, one evidence-based check and a short explanation of your change. Label the response supplied or live, and mark any live test you did not perform. No additional project is required.
